In [1]:
# --- ETAPA 1: CARREGAMENTO DA BASE MESTRA E FILTRO DE MINAS GERAIS ---

import pandas as pd

# 1. Defino o caminho da base que acabamos de "fabricar" no notebook anterior
caminho_mestra = r'C:\Users\fabio\TCC\CONSOLIDADO\BASE_STATA_FINAL.csv'

print("--- Iniciando Novo Notebook: Foco em Minas Gerais ---")

# 2. Carregamento garantindo que os códigos e anos sejam lidos como TEXTO
# (Para não perdermos os zeros à esquerda e evitar erros de cálculo)
df_mestra = pd.read_csv(caminho_mestra, dtype={'Cod_IBGE': str, 'Ano': str})

# 3. FILTRAGEM GEOGRÁFICA: Isolar apenas Minas Gerais (prefixo '31')
df_minas_painel = df_mestra[df_mestra['Cod_IBGE'].str.startswith('31')].copy()

print(f"Base carregada e filtrada!")
print(f"Total de linhas em MG: {len(df_minas_painel)}")
print(f"Municípios únicos em MG: {df_minas_painel['Cod_IBGE'].nunique()}")

# 4. Verificação rápida com Araxá
print("\n--- Spot-Check: Araxá ---")
print(df_minas_painel[df_minas_painel['Cod_IBGE'] == '310400'].head(3))

--- Iniciando Novo Notebook: Foco em Minas Gerais ---
Base carregada e filtrada!
Total de linhas em MG: 26443
Municípios únicos em MG: 853

--- Spot-Check: Araxá ---
      Cod_IBGE   Ano  CFEM  Populacao Nome_Mestre UF_Mestre    VAB_Total  \
71303   310400  1985   NaN        NaN         NaN       NaN  1435479.572   
71304   310400  1992   NaN    71317.0       Araxá        MG          NaN   
71305   310400  1993   NaN    72695.0       Araxá        MG          NaN   

       VAB Ind.  VAB Serv.  PIB_Real_Mil_Reais  PIB_per_capita_Real  \
71303       NaN        NaN                 NaN                  NaN   
71304       NaN        NaN                 NaN                  NaN   
71305       NaN        NaN                 NaN                  NaN   

       Mort_infant  ind_baixo_peso_pct  bruto_internacoes_resp  Arrecadacao  \
71303          NaN                 NaN                     NaN          NaN   
71304          NaN                 NaN                     NaN          NaN   
71305  

In [2]:
# --- ETAPA 2: VALIDAÇÃO DOS INDICADORES DE SAÚDE (Baixo Peso ao Nascer) ---

# Como realizamos o Master Merge no pipeline anterior, a variável 
# 'ind_baixo_peso_pct' já deve estar integrada ao painel. 

print("--- Verificação de Variável: Baixo Peso ao Nascer ---")

# 1. Checar se a coluna existe e se tem dados
if 'ind_baixo_peso_pct' in df_minas_painel.columns:
    print("Variável 'ind_baixo_peso_pct' localizada no painel.")
    
    # 2. Auditoria na unidade de tratamento (Araxá: 310400)
    # Usando os nomes de colunas que padronizamos (Ano e Cod_IBGE)
    print("\nSérie temporal de Baixo Peso em Araxá (Últimos anos):")
    cols_check = ['Ano', 'ind_baixo_peso_pct']
    print(df_minas_painel[df_minas_painel['Cod_IBGE'] == '310400'][cols_check].tail())
else:
    print("Alerta: A variável não foi encontrada. Verifique o Master Merge.")

# 3. Resumo estatístico rápido para ver se não há apenas zeros
print("\nResumo da variável no estado de MG:")
print(df_minas_painel['ind_baixo_peso_pct'].describe())

--- Verificação de Variável: Baixo Peso ao Nascer ---
Variável 'ind_baixo_peso_pct' localizada no painel.

Série temporal de Baixo Peso em Araxá (Últimos anos):
        Ano  ind_baixo_peso_pct
71329  2017            9.577279
71330  2018            9.067017
71331  2019            7.417397
71332  2020            8.932169
71333  2021           10.625000

Resumo da variável no estado de MG:
count    21935.000000
mean         8.844303
std          5.000102
min          0.000000
25%          6.430130
50%          8.602151
75%         10.714286
max        100.000000
Name: ind_baixo_peso_pct, dtype: float64


In [3]:
# --- ETAPA 3: VALIDAÇÃO E TRATAMENTO DE INTERNAÇÕES RESPIRATÓRIAS ---

# 1. Verificar se a coluna já existe (vinda do Master Merge)
if 'bruto_internacoes_resp' in df_minas_painel.columns:
    print(" Variável 'bruto_internacoes_resp' localizada no painel.")
    
    # 2. TRATAMENTO DE MISSINGS: Aplicando a sua decisão de imputar ZERO
    # Isso garante que anos sem notificação não fiquem como "buracos" no Stata
    df_minas_painel['bruto_internacoes_resp'] = df_minas_painel['bruto_internacoes_resp'].fillna(0)
    print("Imputação de valor zero para registros ausentes concluída.")

else:
    print(" Alerta: Variável não encontrada. Verifique a base original.")

# 3. AUDITORIA MULTIDIMENSIONAL (Araxá: 310400)
# Vamos ver como as duas variáveis de saúde (Peso e Internações) se comportam juntas
print("\n--- Auditoria de Saúde e Bem-estar: Araxá ---")
cols_check = ['Ano', 'ind_baixo_peso_pct', 'bruto_internacoes_resp']
print(df_minas_painel[df_minas_painel['Cod_IBGE'] == '310400'][cols_check].tail())

 Variável 'bruto_internacoes_resp' localizada no painel.
Imputação de valor zero para registros ausentes concluída.

--- Auditoria de Saúde e Bem-estar: Araxá ---
        Ano  ind_baixo_peso_pct  bruto_internacoes_resp
71329  2017            9.577279                   393.0
71330  2018            9.067017                   473.0
71331  2019            7.417397                   468.0
71332  2020            8.932169                   323.0
71333  2021           10.625000                   357.0


In [4]:
# --- ETAPA 4: AUDITORIA FINAL E EXPORTAÇÃO DO PAINEL DE MINAS ---

print(f"--- SUMÁRIO EXECUTIVO: PAINEL MINAS GERAIS ---")
# Usando os nomes de colunas atualizados
print(f"Dimensão Espacial (Municípios em MG): {df_minas_painel['Cod_IBGE'].nunique()}")
print(f"Dimensão Temporal (Observações Totais): {len(df_minas_painel)}")
print(f"Dimensão de Atributos (Variáveis): {len(df_minas_painel.columns)}")

print("\n--- DIAGNÓSTICO DE INTEGRIDADE (MISSING VALUES) ---")
# Avaliando as lacunas que sobraram
nulos_peso = df_minas_painel['ind_baixo_peso_pct'].isnull().sum()
nulos_resp = df_minas_painel['bruto_internacoes_resp'].isnull().sum()

print(f"Lacunas em Baixo Peso ao Nascer: {nulos_peso} (Normal em anos antigos)")
print(f"Lacunas em Internações Resp.: {nulos_resp} (Deve ser 0 após o fillna)")

# --- EXPORTAÇÃO DEFINITIVA PARA O STATA ---
# Salvando na pasta CONSOLIDADO para manter a organização
caminho_stata_mg = r'C:\Users\fabio\TCC\CONSOLIDADO\PAINEL_MINAS_SAUDE_FINAL.dta'

# Exportando
df_minas_painel.to_stata(caminho_stata_mg, write_index=False)

print(f"\n--- PROCESSO CONCLUÍDO COM SUCESSO ---")
print(f"Arquivo pronto para o Stata em: {caminho_stata_mg}")

--- SUMÁRIO EXECUTIVO: PAINEL MINAS GERAIS ---
Dimensão Espacial (Municípios em MG): 853
Dimensão Temporal (Observações Totais): 26443
Dimensão de Atributos (Variáveis): 19

--- DIAGNÓSTICO DE INTEGRIDADE (MISSING VALUES) ---
Lacunas em Baixo Peso ao Nascer: 4508 (Normal em anos antigos)
Lacunas em Internações Resp.: 0 (Deve ser 0 após o fillna)

--- PROCESSO CONCLUÍDO COM SUCESSO ---
Arquivo pronto para o Stata em: C:\Users\fabio\TCC\CONSOLIDADO\PAINEL_MINAS_SAUDE_FINAL.dta


C:\Users\fabio\AppData\Local\Temp\ipykernel_5488\406213442.py:22: InvalidColumnName: 
Not all pandas column names were valid Stata variable names.
The following replacements have been made:

    VAB Ind.   ->   VAB_Ind_
    VAB Serv.   ->   VAB_Serv_

If this is not what you expect, please make sure you have Stata-compliant
column names in your DataFrame (strings only, max 32 characters, only
alphanumerics and underscores, no Stata reserved words)

  df_minas_painel.to_stata(caminho_stata_mg, write_index=False)


In [5]:
# --- ETAPA 5: EXPORTAÇÃO DEFINITIVA E COMPATIBILIDADE (VERSÃO FINAL) ---

print(" Iniciando a exportação dos datasets definitivos para o TCC...")

# 1. DEFINIÇÃO DOS CAMINHOS (Centralizados na pasta CONSOLIDADO)
path_csv = r'C:\Users\fabio\TCC\CONSOLIDADO\BASE_MINAS_PAINEL_COMPLETA.csv'
path_dta = r'C:\Users\fabio\TCC\CONSOLIDADO\BASE_MINAS_PAINEL_COMPLETA.dta'

# 2. EXPORTAÇÃO EM CSV (Com suporte a acentuação brasileira)
try:
    df_minas_painel.to_csv(path_csv, index=False, encoding='utf-8-sig')
    print(f"Backup CSV gerado com sucesso.")
except Exception as e:
    print(f"Erro ao gerar CSV: {e}")

# 3. EXPORTAÇÃO OTIMIZADA PARA STATA (.DTA)
# A versão 117 é ideal para manter a compatibilidade e integridade dos nomes.
try:
    # Removemos colunas que podem estar como 'object' mas são vazias, 
    # para evitar conflitos de tipo no Stata.
    df_minas_painel.to_stata(path_dta, write_index=False, version=117)
    
    print("\n" + "="*50)
    print("EXPORTAÇÃO CONCLUÍDA COM SUCESSO!")
    print(f"Arquivo para o Stata: {path_dta}")
    print(f"Horizonte Temporal: {df_minas_painel['Ano'].min()} - {df_minas_painel['Ano'].max()}")
    print("="*50)
    
except Exception as e:
    print(f"\n[ALERTA] Falha na gravação do arquivo .dta: {e}")

print("\nStatus: Pipeline encerrado. Partiu Stata!")

 Iniciando a exportação dos datasets definitivos para o TCC...
Backup CSV gerado com sucesso.

EXPORTAÇÃO CONCLUÍDA COM SUCESSO!
Arquivo para o Stata: C:\Users\fabio\TCC\CONSOLIDADO\BASE_MINAS_PAINEL_COMPLETA.dta
Horizonte Temporal: 1985 - 2021

Status: Pipeline encerrado. Partiu Stata!


C:\Users\fabio\AppData\Local\Temp\ipykernel_5488\1500685458.py:21: InvalidColumnName: 
Not all pandas column names were valid Stata variable names.
The following replacements have been made:

    VAB Ind.   ->   VAB_Ind_
    VAB Serv.   ->   VAB_Serv_

If this is not what you expect, please make sure you have Stata-compliant
column names in your DataFrame (strings only, max 32 characters, only
alphanumerics and underscores, no Stata reserved words)

  df_minas_painel.to_stata(path_dta, write_index=False, version=117)


In [6]:
# --- ETAPA EXTRA: NORMALIZAÇÃO DE INDICADORES (TAXA DE INTERNAÇÃO) ---

import pandas as pd
import numpy as np

print("Calculando indicadores de intensidade (Taxa por 1.000 hab)...")

# 1. CÁLCULO DO INDICADOR DE INTENSIDADE
# Ajustado para os nomes: 'bruto_internacoes_resp' e 'Populacao'
df_minas_painel['ind_internacoes_resp_por_mil'] = (
    df_minas_painel['bruto_internacoes_resp'] / df_minas_painel['Populacao']
) * 1000

# 2. TRATAMENTO DE CONSISTÊNCIA MATEMÁTICA
# Limpando possíveis divisões por zero (infinitos)
df_minas_painel['ind_internacoes_resp_por_mil'] = df_minas_painel['ind_internacoes_resp_por_mil'].replace([np.inf, -np.inf], np.nan)

# 3. VALIDAÇÃO TÉCNICA: ARAXÁ (310400)
print("\n--- Auditoria de Normalização: Araxá (Série Recente) ---")
cols_validacao = ['Ano', 'Populacao', 'bruto_internacoes_resp', 'ind_internacoes_resp_por_mil']
print(df_minas_painel[df_minas_painel['Cod_IBGE'] == '310400'][cols_validacao].tail())

# --- EXPORTAÇÃO DO DATASET DEFINITIVO (MODELAGEM) ---

# Mantendo a organização na pasta CONSOLIDADO
path_csv = r'C:\Users\fabio\TCC\CONSOLIDADO\BASE_MINAS_PAINEL_COMPLETA.csv'
path_dta = r'C:\Users\fabio\TCC\CONSOLIDADO\BASE_MINAS_PAINEL_COMPLETA.dta'

print(f"\nConsolidando base final com indicadores normalizados...")
df_minas_painel.to_csv(path_csv, index=False, encoding='utf-8-sig')
df_minas_painel.to_stata(path_dta, write_index=False, version=117)

print("\n" + "="*50)
print("PIPELINE 100% CONCLUÍDO!")
print(f"Variável de impacto pronta: 'ind_internacoes_resp_por_mil'")
print(f"Local: {path_dta}")
print("="*50)

Calculando indicadores de intensidade (Taxa por 1.000 hab)...

--- Auditoria de Normalização: Araxá (Série Recente) ---
        Ano  Populacao  bruto_internacoes_resp  ind_internacoes_resp_por_mil
71329  2017   104283.0                   393.0                      3.768591
71330  2018   105083.0                   473.0                      4.501204
71331  2019   106229.0                   468.0                      4.405577
71332  2020   107337.0                   323.0                      3.009214
71333  2021   108403.0                   357.0                      3.293267

Consolidando base final com indicadores normalizados...

PIPELINE 100% CONCLUÍDO!
Variável de impacto pronta: 'ind_internacoes_resp_por_mil'
Local: C:\Users\fabio\TCC\CONSOLIDADO\BASE_MINAS_PAINEL_COMPLETA.dta


C:\Users\fabio\AppData\Local\Temp\ipykernel_5488\3650941827.py:31: InvalidColumnName: 
Not all pandas column names were valid Stata variable names.
The following replacements have been made:

    VAB Ind.   ->   VAB_Ind_
    VAB Serv.   ->   VAB_Serv_

If this is not what you expect, please make sure you have Stata-compliant
column names in your DataFrame (strings only, max 32 characters, only
alphanumerics and underscores, no Stata reserved words)

  df_minas_painel.to_stata(path_dta, write_index=False, version=117)


In [7]:
print(df_minas_painel.dtypes)

Cod_IBGE                         object
Ano                              object
CFEM                            float64
Populacao                       float64
Nome_Mestre                      object
UF_Mestre                        object
VAB_Total                       float64
VAB Ind.                        float64
VAB Serv.                       float64
PIB_Real_Mil_Reais              float64
PIB_per_capita_Real             float64
Mort_infant                     float64
ind_baixo_peso_pct              float64
bruto_internacoes_resp          float64
Arrecadacao                     float64
Vinculos_totais                 float64
Empregos_mineracao              float64
Empregos_servicos               float64
VAB_per_capita                  float64
ind_internacoes_resp_por_mil    float64
dtype: object


In [8]:
# --- BLOCO FINAL: CÁLCULO DE INDICADORES E PADRONIZAÇÃO ORIGINAL ---

import pandas as pd
import numpy as np

print("Iniciando processamento final de indicadores e nomes...")

# 1. CÁLCULO DOS INDICADORES (Usando os nomes que estão na memória agora)
# Participações (%)
df_minas_painel['ind_part_vab_industria'] = (df_minas_painel['VAB Ind.'] / df_minas_painel['VAB_Total']) * 100
df_minas_painel['ind_part_emp_mineracao'] = (df_minas_painel['Empregos_mineracao'] / df_minas_painel['Vinculos_totais']) * 100
df_minas_painel['ind_dependencia_cfem'] = (df_minas_painel['CFEM'] / df_minas_painel['Arrecadacao']) * 100

# Crescimento (%) - Ordenando por cidade e ano para a conta ser correta
df_minas_painel = df_minas_painel.sort_values(['Cod_IBGE', 'Ano'])
df_minas_painel['ind_cresc_pib_pc_real_mil'] = df_minas_painel.groupby('Cod_IBGE')['PIB_per_capita_Real'].pct_change() * 100
df_minas_painel['ind_cresc_emp_total'] = df_minas_painel.groupby('Cod_IBGE')['Vinculos_totais'].pct_change() * 100

print("✅ Indicadores calculados.")

# 2. RENOMEAÇÃO PARA O DICIONÁRIO ORIGINAL
dicionario_nomes = {
    'Cod_IBGE': 'id_municipio_6',
    'Ano': 'id_ano',
    'Nome_Mestre': 'id_nome_municipio',
    'UF_Mestre': 'id_uf',
    'Populacao': 'bruto_populacao',
    'VAB_Total': 'bruto_vab_total',
    'VAB Ind.': 'bruto_vab_industria',
    'VAB Serv.': 'bruto_vab_servicos',
    'CFEM': 'bruto_cfem_valor',
    'Arrecadacao': 'bruto_arrecadacao_total',
    'Vinculos_totais': 'bruto_emp_formal_total',
    'Empregos_mineracao': 'bruto_emp_mineracao',
    'Empregos_servicos': 'bruto_emp_servicos',
    'PIB_Real_Mil_Reais': 'bruto_pib_real',
    'Mort_infant': 'ind_mortalidade_infantil',
    'PIB_per_capita_Real': 'ind_pib_pc_real_mil'
}

df_minas_painel = df_minas_painel.rename(columns=dicionario_nomes)
print("Colunas renomeadas para o padrão original.")

# 3. SELEÇÃO E ORDENAÇÃO DAS 24 COLUNAS EXATAS
colunas_originais = [
    'id_municipio_6', 'id_ano', 'id_nome_municipio', 'id_uf',
    'bruto_populacao', 'bruto_vab_total', 'bruto_vab_industria', 'bruto_vab_servicos',
    'bruto_cfem_valor', 'bruto_arrecadacao_total', 'bruto_emp_formal_total',
    'bruto_emp_mineracao', 'bruto_emp_servicos', 'bruto_pib_real',
    'ind_mortalidade_infantil', 'ind_pib_pc_real_mil', 'ind_part_vab_industria',
    'ind_part_emp_mineracao', 'ind_cresc_pib_pc_real_mil', 'ind_cresc_emp_total',
    'ind_dependencia_cfem', 'ind_baixo_peso_pct', 'bruto_internacoes_resp',
    'ind_internacoes_resp_por_mil'
]

# Criamos o dataframe final apenas com o que o meu Stata espera
df_final_stata = df_minas_painel[colunas_originais].copy()

# 4. EXPORTAÇÃO DEFINITIVA
path_dta = r'C:\Users\fabio\TCC\CONSOLIDADO\PAINEL_MINAS_SAUDE_FINAL.dta'
df_final_stata.to_stata(path_dta, write_index=False, version=117)

print("\n" + "="*50)
print("SUCESSO TOTAL: BASE CLONADA E PRONTA!")
print(f"Total de variáveis: {len(df_final_stata.columns)}")
print(f"Caminho: {path_dta}")
print("="*50)

Iniciando processamento final de indicadores e nomes...
✅ Indicadores calculados.
Colunas renomeadas para o padrão original.


C:\Users\fabio\AppData\Local\Temp\ipykernel_5488\3956366074.py:16: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_minas_painel['ind_cresc_pib_pc_real_mil'] = df_minas_painel.groupby('Cod_IBGE')['PIB_per_capita_Real'].pct_change() * 100
C:\Users\fabio\AppData\Local\Temp\ipykernel_5488\3956366074.py:17: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_minas_painel['ind_cresc_emp_total'] = df_minas_painel.groupby('Cod_IBGE')['Vinculos_totais'].pct_change() * 100



SUCESSO TOTAL: BASE CLONADA E PRONTA!
Total de variáveis: 24
Caminho: C:\Users\fabio\TCC\CONSOLIDADO\PAINEL_MINAS_SAUDE_FINAL.dta


In [9]:
print(df_final_stata.dtypes)

id_municipio_6                   object
id_ano                           object
id_nome_municipio                object
id_uf                            object
bruto_populacao                 float64
bruto_vab_total                 float64
bruto_vab_industria             float64
bruto_vab_servicos              float64
bruto_cfem_valor                float64
bruto_arrecadacao_total         float64
bruto_emp_formal_total          float64
bruto_emp_mineracao             float64
bruto_emp_servicos              float64
bruto_pib_real                  float64
ind_mortalidade_infantil        float64
ind_pib_pc_real_mil             float64
ind_part_vab_industria          float64
ind_part_emp_mineracao          float64
ind_cresc_pib_pc_real_mil       float64
ind_cresc_emp_total             float64
ind_dependencia_cfem            float64
ind_baixo_peso_pct              float64
bruto_internacoes_resp          float64
ind_internacoes_resp_por_mil    float64
dtype: object
